# DynaLoRA on a larger base model: Qwen2.5-7B in 4 bits (Colab, GPU)

Trains the balanced IMDB and PubMed adapters on Qwen2.5-7B with the base model quantized to 4-bit NF4 (QLoRA, bitsandbytes; same LoRA recipe as the other runs: rank 8, alpha 16, lr 2e-4, 3 epochs), then runs the teacher-forced likelihood, the zero-shot benchmarks, a reduced generation grid (DynaLoRA and weight-space scaling at lambda 0, 0.5, 1, 1.5, 2 and 3), re-scoring with SmolLM2-1.7B and MAUVE. The base model is loaded in 4 bits in every step, as the adapters were trained against it. Results go to `/content/drive/MyDrive/dynalora_results/qwen7b`, and every step skips what already exists, so the notebook can be re-run after a disconnection. An A100 or L4 is recommended (bfloat16 compute); on a T4 the run takes several times longer.

In [ ]:
!nvidia-smi -L
!git clone -b paper https://github.com/alexroehrs/dynalora.git /content/dynalora
%cd /content/dynalora
!git pull -q
!pip install -q -r requirements.txt lm-eval bitsandbytes mauve-text faiss-cpu
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/dynalora_results/qwen7b', exist_ok=True)
print(os.listdir('/content/drive/MyDrive/dynalora_results/qwen7b'))

## 1. Train the adapters (4-bit base model)

In [ ]:
!python scripts/run_all.py --out /content/drive/MyDrive/dynalora_results/qwen7b --models Qwen/Qwen2.5-7B --configs imdb_bal pubmed --load_in_4bit --train_only
!head -n 1 /content/drive/MyDrive/dynalora_results/qwen7b/adapters/*/train_log.txt

## 2. Teacher-forced likelihood on domain and general text

In [ ]:
!python scripts/likelihood_tradeoff.py --model Qwen/Qwen2.5-7B --adapters_dir /content/drive/MyDrive/dynalora_results/qwen7b/adapters --configs imdb_bal pubmed --load_in_4bit --batch_size 4 --out /content/drive/MyDrive/dynalora_results/qwen7b/likelihood
!python scripts/likelihood_summary.py /content/drive/MyDrive/dynalora_results/qwen7b/likelihood > /dev/null

## 3. Zero-shot benchmarks

In [ ]:
!python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-7B --adapters_dir /content/drive/MyDrive/dynalora_results/qwen7b/adapters --configs imdb_bal pubmed --load_in_4bit --batch_size 4 --limit 500 --lambdas 0 0.5 1 1.5 2 2.5 3 --out /content/drive/MyDrive/dynalora_results/qwen7b/bench

## 4. Reduced generation grid: DynaLoRA and weight-space scaling

In [ ]:
!python scripts/run_all.py --out /content/drive/MyDrive/dynalora_results/qwen7b --models Qwen/Qwen2.5-7B --configs imdb_bal pubmed --load_in_4bit --methods dynalora weight --lambdas 0 0.5 1 1.5 2 3 --clf_configs imdb_bal yelp pubmed --n_samples 20

## 5. Re-scoring with SmolLM2-1.7B and MAUVE

In [ ]:
!python scripts/rescore.py --inputs /content/drive/MyDrive/dynalora_results/qwen7b/generations_all_metrics.csv --out /content/drive/MyDrive/dynalora_results/qwen7b/rescore
!python scripts/mauve_eval.py --inputs /content/drive/MyDrive/dynalora_results/qwen7b/generations_all_metrics.csv --out /content/drive/MyDrive/dynalora_results/qwen7b/mauve

## 6. Package the new results (no adapters, no reference text)

In [ ]:
!cd /content/drive/MyDrive/dynalora_results/qwen7b && zip -qr /content/dynalora_qwen7b.zip summary.csv entropy_matched.csv run_info.json generations_all_metrics.csv likelihood bench rescore mauve adapters/*/train_log.txt
from google.colab import files
files.download('/content/dynalora_qwen7b.zip')